# DPI Probe Gate: GPU smoke test (Kaggle)

Launch from the CLI: `kaggle kernels push -p .` (uses `kernel-metadata.json`). From the UI instead: **Settings > Accelerator > GPU T4 x2**, **Settings > Internet > On**, and add the code dataset as an input.
If a model download is refused, add your Hugging Face token: **Add-ons > Secrets**, name it `HF_TOKEN`.

This runs 5 English scenarios per model and checks three things:
1. Model loads and generates.
2. The raw output parses into tool calls.
3. Seconds per scenario (x840 = one full run).

In [ ]:
# The GitHub repo is private, so the code comes in as a private Kaggle dataset
# (dataset_sources in kernel-metadata.json, or Add Input in the UI).
import glob, os, shutil, zipfile
!nvidia-smi -L
W = "/kaggle/working/safl-hackathon"
hits = glob.glob("/kaggle/input/**/collect.py", recursive=True)
if hits:
    shutil.copytree(os.path.dirname(hits[0]), W, dirs_exist_ok=True)
else:
    zips = glob.glob("/kaggle/input/**/*.zip", recursive=True)
    assert zips, "code dataset not attached: nothing under /kaggle/input"
    zipfile.ZipFile(zips[0]).extractall(W)
%cd {W}
!ls -R | head -40

In [ ]:
# Gemma-4 (dhee-pravega) needs a recent transformers; 5.7.0 is the version checked locally.
!pip install -q "transformers==5.7.0" accelerate
import transformers, torch
print(transformers.__version__, torch.__version__, torch.cuda.device_count(), "GPUs")

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN set from Kaggle secrets")
except Exception as e:
    print("No HF_TOKEN secret; public downloads only:", type(e).__name__)

In [ ]:
# Logs go to files because `kaggle kernels logs` can return nothing; outputs always download.
!mkdir -p /kaggle/working/logs && python collect.py --model dhee-pravega --langs en --limit 5 2>&1 | tee /kaggle/working/logs/dhee-pravega.log

In [ ]:
!python collect.py --model qwen3-8b --langs en --limit 5 2>&1 | tee /kaggle/working/logs/qwen3-8b.log

In [ ]:
# One line per scenario. Look for: empty or garbled output (fp16 overflow on T4),
# and outputs that clearly call a tool but show calls=[] (parser needs that format).
import json, numpy as np
for m in ["qwen3-8b", "dhee-pravega"]:
    p = f"data/{m}/en_smoke_meta.jsonl"
    if not os.path.exists(p):
        print(m, ": no output (run failed?)"); continue
    acts = np.load(f"data/{m}/en_smoke_acts.npy")
    print(f"== {m}  acts {acts.shape}  any NaN/inf: {not np.isfinite(acts.astype(np.float32)).all()}")
    for r in map(json.loads, open(p, encoding="utf-8")):
        print(r["id"], "| calls:", r["calls"], "| executed_target:", r["executed_target"])
        print("   raw:", repr(r["output"][:300]))

In [ ]:
# Bundle the smoke outputs (kaggle kernels output <kernel> --file-pattern smoke_outputs).
!cd /kaggle/working/safl-hackathon && zip -q -r /kaggle/working/smoke_outputs.zip data/qwen3-8b data/dhee-pravega && ls -la /kaggle/working/smoke_outputs.zip